In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/Users/pranitac/Desktop/Behavioral-nudge/Data/halfhourly_dataset/block_0.csv")

df.head()

,LCLid,tstp,energy(kWh/hh)
0,MAC000002,2012-10-12 00:30:00.0000000,0
1,MAC000002,2012-10-12 01:00:00.0000000,0
2,MAC000002,2012-10-12 01:30:00.0000000,0
3,MAC000002,2012-10-12 02:00:00.0000000,0
4,MAC000002,2012-10-12 02:30:00.0000000,0


In [3]:
df.columns

Index(['LCLid', 'tstp', 'energy(kWh/hh)'], dtype='object')

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1222670 entries, 0 to 1222669
Data columns (total 3 columns):
 #   Column          Non-Null Count    Dtype 
---  ------          --------------    ----- 
 0   LCLid           1222670 non-null  object
 1   tstp            1222670 non-null  object
 2   energy(kWh/hh)  1222670 non-null  object
dtypes: object(3)
memory usage: 28.0+ MB


In [5]:
df.describe()

,LCLid,tstp,energy(kWh/hh)
count,1222670,1222670,1222670
unique,50,39292,5022
top,MAC000246,2012-12-16 00:00:00.0000000,0.013
freq,39245,50,6238


In [6]:
# Rename columns (cleaner names)
df = df.rename(columns={
    'LCLid': 'user_id',
    'tstp': 'timestamp',
    'energy(kWh/hh)': 'energy_kwh'
})

# Convert timestamp
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Convert energy to numeric
df['energy_kwh'] = pd.to_numeric(df['energy_kwh'], errors='coerce')

# Check again
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1222670 entries, 0 to 1222669
Data columns (total 3 columns):
 #   Column      Non-Null Count    Dtype         
---  ------      --------------    -----         
 0   user_id     1222670 non-null  object        
 1   timestamp   1222670 non-null  datetime64[ns]
 2   energy_kwh  1222620 non-null  float64       
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 28.0+ MB


In [7]:
df['hour'] = df['timestamp'].dt.hour
df['date'] = df['timestamp'].dt.date

In [8]:
df['is_peak'] = df['hour'].between(17, 20)

In [9]:
df.head()

,user_id,timestamp,energy_kwh,hour,date,is_peak
0,MAC000002,2012-10-12 00:30:00,0.0,0,2012-10-12,False
1,MAC000002,2012-10-12 01:00:00,0.0,1,2012-10-12,False
2,MAC000002,2012-10-12 01:30:00,0.0,1,2012-10-12,False
3,MAC000002,2012-10-12 02:00:00,0.0,2,2012-10-12,False
4,MAC000002,2012-10-12 02:30:00,0.0,2,2012-10-12,False


In [10]:
daily = (
    df.groupby(['user_id', 'date'])
      .agg(
          total_kwh=('energy_kwh', 'sum'),
          peak_kwh=('energy_kwh', lambda x: x[df.loc[x.index, 'is_peak']].sum()),
          offpeak_kwh=('energy_kwh', lambda x: x[~df.loc[x.index, 'is_peak']].sum())
      )
      .reset_index()
)

daily['offpeak_share'] = daily['offpeak_kwh'] / daily['total_kwh']
daily.head()

,user_id,date,total_kwh,peak_kwh,offpeak_kwh,offpeak_share
0,MAC000002,2012-10-12,7.098,3.124,3.974,0.559876
1,MAC000002,2012-10-13,11.087,2.756,8.331,0.751421
2,MAC000002,2012-10-14,13.223,5.532,7.691,0.581638
3,MAC000002,2012-10-15,10.257,3.333,6.924,0.675051
4,MAC000002,2012-10-16,9.769,2.759,7.010,0.717576


In [11]:
daily = daily[daily['total_kwh'] > 0].copy()

daily.head()
daily.describe()

,total_kwh,peak_kwh,offpeak_kwh,offpeak_share
count,25574.000000,25574.000000,25574.000000,25574.000000
mean,21.543821,5.208460,16.335361,0.757011
std,20.205007,4.802661,15.882727,0.083866
min,0.012000,0.000000,0.012000,0.348452
25%,10.231000,2.176250,7.598750,0.705166
50%,15.260000,3.697000,11.410500,0.765182
75%,25.332750,6.613750,18.970500,0.817323
max,277.973999,51.522000,227.812999,1.000000


In [12]:
daily.describe()

,total_kwh,peak_kwh,offpeak_kwh,offpeak_share
count,25574.000000,25574.000000,25574.000000,25574.000000
mean,21.543821,5.208460,16.335361,0.757011
std,20.205007,4.802661,15.882727,0.083866
min,0.012000,0.000000,0.012000,0.348452
25%,10.231000,2.176250,7.598750,0.705166
50%,15.260000,3.697000,11.410500,0.765182
75%,25.332750,6.613750,18.970500,0.817323
max,277.973999,51.522000,227.812999,1.000000


In [13]:
# Define carbon intensity
peak_intensity = 450   # gCO2 per kWh
offpeak_intensity = 250

# Calculate emissions
daily['co2_emissions'] = (
    daily['peak_kwh'] * peak_intensity +
    daily['offpeak_kwh'] * offpeak_intensity
)

daily.head()
daily.describe()

,total_kwh,peak_kwh,offpeak_kwh,offpeak_share,co2_emissions
count,25574.000000,25574.000000,25574.000000,25574.000000,25574.000000
mean,21.543821,5.208460,16.335361,0.757011,6427.647249
std,20.205007,4.802661,15.882727,0.083866,5948.958446
min,0.012000,0.000000,0.012000,0.348452,3.000000
25%,10.231000,2.176250,7.598750,0.705166,3039.850000
50%,15.260000,3.697000,11.410500,0.765182,4578.575000
75%,25.332750,6.613750,18.970500,0.817323,7615.487470
max,277.973999,51.522000,227.812999,1.000000,79525.699910


In [14]:
import numpy as np

# Get unique users
users = daily['user_id'].unique()

# Random assignment
np.random.seed(42)

treatment_map = pd.DataFrame({
    'user_id': users,
    'variant': np.random.choice(['control', 'treatment'], size=len(users))
})

# Merge back
daily = daily.merge(treatment_map, on='user_id')

daily.head()

,user_id,date,total_kwh,peak_kwh,offpeak_kwh,offpeak_share,co2_emissions,variant
0,MAC000002,2012-10-12,7.098,3.124,3.974,0.559876,2399.30,control
1,MAC000002,2012-10-13,11.087,2.756,8.331,0.751421,3322.95,control
2,MAC000002,2012-10-14,13.223,5.532,7.691,0.581638,4412.15,control
3,MAC000002,2012-10-15,10.257,3.333,6.924,0.675051,3230.85,control
4,MAC000002,2012-10-16,9.769,2.759,7.010,0.717576,2994.05,control


In [15]:
# Copy original values
daily['peak_kwh_adj'] = daily['peak_kwh']
daily['offpeak_kwh_adj'] = daily['offpeak_kwh']

# Apply effect ONLY to treatment group
effect_size = 0.10  # 10% shift

mask = daily['variant'] == 'treatment'

shift = daily.loc[mask, 'peak_kwh'] * effect_size

daily.loc[mask, 'peak_kwh_adj'] -= shift
daily.loc[mask, 'offpeak_kwh_adj'] += shift

In [16]:
# New totals (should remain same)
daily['total_kwh_adj'] = daily['peak_kwh_adj'] + daily['offpeak_kwh_adj']

# New off-peak share
daily['offpeak_share_adj'] = daily['offpeak_kwh_adj'] / daily['total_kwh_adj']

# New CO2 emissions
daily['co2_emissions_adj'] = (
    daily['peak_kwh_adj'] * 450 +
    daily['offpeak_kwh_adj'] * 250
)

In [17]:
daily.groupby('variant')[['co2_emissions', 'co2_emissions_adj']].mean()

,co2_emissions,co2_emissions_adj
variant,,
control,5691.259625,5691.259625
treatment,7019.634958,6907.979327


In [18]:
daily.groupby('variant').apply(
    lambda x: ((x['co2_emissions'] - x['co2_emissions_adj']) / x['co2_emissions']).mean()
)

variant
control      0.000000
treatment    0.015779
dtype: float64

In [19]:
from scipy.stats import ttest_ind

control = daily[daily['variant'] == 'control']['co2_emissions_adj']
treatment = daily[daily['variant'] == 'treatment']['co2_emissions_adj']

t_stat, p_value = ttest_ind(control, treatment)

print("t-stat:", t_stat)
print("p-value:", p_value)

t-stat: -16.538170032034024
p-value: 4.049353631795766e-61


## SQL Transformation
We use SQL to replicate the daily aggregation step, demonstrating how similar transformations would be performed in a production analytics environment.

In [20]:
import sqlite3

conn = sqlite3.connect("energy.db")

df.to_sql("energy", conn, if_exists="replace", index=False)

print("Table saved to SQLite!")

Table saved to SQLite!


In [21]:
query = """
SELECT name
FROM sqlite_master
WHERE type='table';
"""

pd.read_sql_query(query, conn)

,name
0,energy


In [22]:
query = """
SELECT *
FROM energy
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,user_id,timestamp,energy_kwh,hour,date,is_peak
0,MAC000002,2012-10-12 00:30:00,0.0,0,2012-10-12,0
1,MAC000002,2012-10-12 01:00:00,0.0,1,2012-10-12,0
2,MAC000002,2012-10-12 01:30:00,0.0,1,2012-10-12,0
3,MAC000002,2012-10-12 02:00:00,0.0,2,2012-10-12,0
4,MAC000002,2012-10-12 02:30:00,0.0,2,2012-10-12,0


In [23]:
query = """
SELECT
    user_id,
    date,
    SUM(energy_kwh) AS total_kwh,
    SUM(CASE WHEN is_peak = 1 THEN energy_kwh ELSE 0 END) AS peak_kwh,
    SUM(CASE WHEN is_peak = 0 THEN energy_kwh ELSE 0 END) AS offpeak_kwh
FROM energy
GROUP BY user_id, date
ORDER BY user_id, date;
"""

daily_sql = pd.read_sql_query(query, conn)
daily_sql.head()

,user_id,date,total_kwh,peak_kwh,offpeak_kwh
0,MAC000002,2012-10-12,7.098,3.124,3.974
1,MAC000002,2012-10-13,11.087,2.756,8.331
2,MAC000002,2012-10-14,13.223,5.532,7.691
3,MAC000002,2012-10-15,10.257,3.333,6.924
4,MAC000002,2012-10-16,9.769,2.759,7.010


In [24]:
daily_sql['offpeak_share'] = daily_sql['offpeak_kwh'] / daily_sql['total_kwh']
daily_sql = daily_sql[daily_sql['total_kwh'] > 0].copy()

daily_sql.head()

,user_id,date,total_kwh,peak_kwh,offpeak_kwh,offpeak_share
0,MAC000002,2012-10-12,7.098,3.124,3.974,0.559876
1,MAC000002,2012-10-13,11.087,2.756,8.331,0.751421
2,MAC000002,2012-10-14,13.223,5.532,7.691,0.581638
3,MAC000002,2012-10-15,10.257,3.333,6.924,0.675051
4,MAC000002,2012-10-16,9.769,2.759,7.010,0.717576


In [25]:
daily.head()

,user_id,date,total_kwh,peak_kwh,offpeak_kwh,offpeak_share,co2_emissions,variant,peak_kwh_adj,offpeak_kwh_adj,total_kwh_adj,offpeak_share_adj,co2_emissions_adj
0,MAC000002,2012-10-12,7.098,3.124,3.974,0.559876,2399.30,control,3.124,3.974,7.098,0.559876,2399.30
1,MAC000002,2012-10-13,11.087,2.756,8.331,0.751421,3322.95,control,2.756,8.331,11.087,0.751421,3322.95
2,MAC000002,2012-10-14,13.223,5.532,7.691,0.581638,4412.15,control,5.532,7.691,13.223,0.581638,4412.15
3,MAC000002,2012-10-15,10.257,3.333,6.924,0.675051,3230.85,control,3.333,6.924,10.257,0.675051,3230.85
4,MAC000002,2012-10-16,9.769,2.759,7.010,0.717576,2994.05,control,2.759,7.010,9.769,0.717576,2994.05


We use SQL to aggregate half-hourly energy data into daily household-level metrics, including total consumption, peak usage, and off-peak usage.

This mirrors how transformations would be performed in a production data warehouse.

SELECT
    user_id,
    date,
    SUM(energy_kwh) AS total_kwh,
    SUM(CASE WHEN is_peak = 1 THEN energy_kwh ELSE 0 END) AS peak_kwh,
    SUM(CASE WHEN is_peak = 0 THEN energy_kwh ELSE 0 END) AS offpeak_kwh
FROM energy
GROUP BY user_id, date;

In [26]:
peak_intensity = 450
offpeak_intensity = 250

daily_sql['co2_emissions'] = (
    daily_sql['peak_kwh'] * peak_intensity +
    daily_sql['offpeak_kwh'] * offpeak_intensity
)

In [27]:
import numpy as np

users = daily_sql['user_id'].unique()

np.random.seed(42)

treatment_map = pd.DataFrame({
    'user_id': users,
    'variant': np.random.choice(['control', 'treatment'], size=len(users))
})

daily_sql = daily_sql.merge(treatment_map, on='user_id')

In [28]:
daily_sql['peak_kwh_adj'] = daily_sql['peak_kwh']
daily_sql['offpeak_kwh_adj'] = daily_sql['offpeak_kwh']

effect_size = 0.10

mask = daily_sql['variant'] == 'treatment'

shift = daily_sql.loc[mask, 'peak_kwh'] * effect_size

daily_sql.loc[mask, 'peak_kwh_adj'] -= shift
daily_sql.loc[mask, 'offpeak_kwh_adj'] += shift

In [29]:
daily_sql['total_kwh_adj'] = daily_sql['peak_kwh_adj'] + daily_sql['offpeak_kwh_adj']

daily_sql['offpeak_share_adj'] = (
    daily_sql['offpeak_kwh_adj'] / daily_sql['total_kwh_adj']
)

daily_sql['co2_emissions_adj'] = (
    daily_sql['peak_kwh_adj'] * 450 +
    daily_sql['offpeak_kwh_adj'] * 250
)

In [30]:
daily_sql.groupby('variant')[['co2_emissions', 'co2_emissions_adj']].mean()

,co2_emissions,co2_emissions_adj
variant,,
control,5691.259625,5691.259625
treatment,7019.634958,6907.979327


In [31]:
from scipy.stats import ttest_ind

control = daily_sql[daily_sql['variant'] == 'control']['co2_emissions_adj']
treatment = daily_sql[daily_sql['variant'] == 'treatment']['co2_emissions_adj']

t_stat, p_value = ttest_ind(control, treatment)

print("t-stat:", t_stat)
print("p-value:", p_value)


t-stat: -16.538170032034035
p-value: 4.049353631794957e-61
